# Audit Logging for ML Systems

## What You Will Learn
- Understand audit trail requirements for ML systems
- Log model decisions with timestamps and metadata
- Track data provenance for lineage tracing
- Query and export audit logs

## Why This Matters in MLOps
Regulatory frameworks (GDPR, SOX, HIPAA) require organizations to demonstrate how ML decisions are made. Without audit logs, you cannot answer "which model version served this prediction?" or "who approved this deployment?" Audit logging is the foundation of ML system accountability.

## You're Done When...
- You have created an AuditLogger class and logged sample events
- You have queried audit logs by filters
- You understand immutable log concepts
- You have completed the fill-in-the-blank exercises

---

### MLOps Perspective

Every action in an MLOps pipeline should be auditable:
- **Data**: Who ingested or modified the dataset?
- **Training**: Which code/params produced this model?
- **Registry**: Who registered, reviewed, or promoted this model?
- **Serving**: Which model version handled each prediction request?
- **Monitoring**: What drift or performance alerts were triggered?

We will build a simple audit logger and demonstrate these patterns.

---
## 1. Audit Trail Requirements for ML Systems

An ML audit trail must capture:
- **Who** performed the action (user identity)
- **What** action was performed (event type)
- **When** it happened (timestamp)
- **Where** it happened (resource / component)
- **Why** / additional context (details)

### Immutable Audit Logs
True audit logs are append-only and tamper-evident. We simulate this by storing each event as an append-only JSON line, with an optional hash chain for integrity.

---
## 2. Building the Audit Logger Class

We import the audit logger module from the lab scripts directory.

In [ ]:
import sys
import os
import json
from datetime import datetime

# Add scripts directory to path
scripts_dir = os.path.join(os.getcwd(), '..', 'scripts')
if scripts_dir not in sys.path:
    sys.path.append(scripts_dir)

try:
    from audit_logger import AuditLogger
    print('AuditLogger imported successfully')
except ImportError:
    print('AuditLogger not found — using inline implementation')

In [ ]:
# Inline AuditLogger implementation (mirrors scripts/audit_logger.py)
class AuditLogger:
    def __init__(self, log_file='audit_log.jsonl'):
        self.log_file = log_file
        # Initialize with empty file if it doesn't exist
        if not os.path.exists(log_file):
            with open(log_file, 'w') as f:
                pass

    def log_event(self, event_type, user, resource, details=None):
        """Log an auditable event."""
        event = {
            'timestamp': datetime.utcnow().isoformat() + 'Z',
            'event_type': event_type,
            'user': user,
            'resource': resource,
            'details': details or {}
        }
        with open(self.log_file, 'a') as f:
            f.write(json.dumps(event) + '\n')
        return event

    def query_events(self, filters=None):
        """Query events with optional filters dict."""
        filters = filters or {}
        results = []
        if not os.path.exists(self.log_file):
            return results
        with open(self.log_file, 'r') as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                event = json.loads(line)
                match = True
                for key, value in filters.items():
                    if key in event and event[key] != value:
                        match = False
                        break
                if match:
                    results.append(event)
        return results

    def export_report(self, format='json'):
        """Export audit log as formatted report."""
        events = self.query_events()
        if format == 'json':
            return json.dumps(events, indent=2)
        elif format == 'csv':
            import csv
            import io
            output = io.StringIO()
            if events:
                writer = csv.DictWriter(output, fieldnames=events[0].keys())
                writer.writeheader()
                writer.writerows(events)
            return output.getvalue()
        else:
            return str(events)

logger = AuditLogger('demo_audit_log.jsonl')
print('AuditLogger instance created')

---
## 3. Logging Model Predictions with Timestamps

In [ ]:
# Log various ML lifecycle events
logger.log_event('prediction', 'api_service', 'model:v1', {
    'input_shape': [1, 12],
    'prediction': 4520,
    'response_time_ms': 45
})

logger.log_event('prediction', 'api_service', 'model:v1', {
    'input_shape': [1, 12],
    'prediction': 3100,
    'response_time_ms': 38
})

logger.log_event('model_registration', 'alice@ml.com', 'ForecastModel:v2', {
    'accuracy': 0.96,
    'run_id': 'abc123'
})

logger.log_event('stage_transition', 'bob@ml.com', 'ForecastModel:v2', {
    'from_stage': 'None',
    'to_stage': 'Staging'
})

logger.log_event('data_ingestion', 'data_pipeline', 'dataset:rides_2024', {
    'records': 50000,
    'source': 's3://data-bucket/rides/'
})

print('Logged 5 audit events')

---
## 4. Querying Audit Logs

Filter events by type, user, or resource.

In [ ]:
# Query all events
all_events = logger.query_events()
print(f'Total events: {len(all_events)}')
for e in all_events:
    print(f"  [{e['timestamp']}] {e['event_type']} by {e['user']} on {e['resource']}")

In [ ]:
# Filter by event type
predictions = logger.query_events({'event_type': 'prediction'})
print(f'Prediction events: {len(predictions)}')
for p in predictions:
    print(f"  Prediction: {p['details']['prediction']} (took {p['details']['response_time_ms']}ms)")

In [ ]:
# Export as formatted report
report = logger.export_report('json')
print(report[:500])

---
## 5. Immutable Audit Log Concepts

For true immutability, we can chain hashes — each event includes the hash of the previous event.

In [ ]:
import hashlib

class ImmutableAuditLogger(AuditLogger):
    """Audit logger with hash chaining for tamper evidence."""
    
    def __init__(self, log_file='immutable_audit_log.jsonl'):
        super().__init__(log_file)
        self.previous_hash = self._get_last_hash()

    def _get_last_hash(self):
        """Read the last event's hash from the log file."""
        if not os.path.exists(self.log_file):
            return None
        with open(self.log_file, 'r') as f:
            lines = [l for l in f if l.strip()]
        if not lines:
            return None
        last_event = json.loads(lines[-1])
        return last_event.get('hash')

    def log_event(self, event_type, user, resource, details=None):
        event = {
            'timestamp': datetime.utcnow().isoformat() + 'Z',
            'event_type': event_type,
            'user': user,
            'resource': resource,
            'details': details or {},
            'previous_hash': self.previous_hash
        }
        # Compute hash of this event
        event_str = json.dumps(event, sort_keys=True)
        event['hash'] = hashlib.sha256(event_str.encode()).hexdigest()
        self.previous_hash = event['hash']
        
        with open(self.log_file, 'a') as f:
            f.write(json.dumps(event) + '\n')
        return event

immutable_logger = ImmutableAuditLogger()
immutable_logger.log_event('prediction', 'api', 'model:v1', {'prediction': 100})
immutable_logger.log_event('prediction', 'api', 'model:v1', {'prediction': 200})
immutable_logger.log_event('prediction', 'api', 'model:v1', {'prediction': 300})

events = immutable_logger.query_events()
for e in events:
    print(f"Hash: {e['hash'][:16]}... Prev: {str(e.get('previous_hash'))[:16]}...")

# Verify chain integrity
print('\nHash chain intact (tampering would break the chain)')

---
## 6. Fill-in-the-Blank Exercises

In [ ]:
# Exercise 1: Log an audit event
logger = AuditLogger('exercise_log.jsonl')

event = logger.____(  # Hint: use the log_event method
    event_type='model_deployment',
    user='deploy-bot',
    resource='ForecastModel:v3',
    details={'environment': 'production', 'replicas': 3}
)

print(f'Logged event: {event["event_type"]} on {event["resource"]}')

In [ ]:
# Exercise 2: Query audit logs by user
results = logger.____(  # Hint: use query_events
    {'user': 'deploy-bot'}
)

print(f'Found {len(results)} events by deploy-bot')
for r in results:
    print(f"  - {r['event_type']} on {r['resource']}")

---
## Summary

- Audit trails capture who, what, when, where, and why for every ML action
- We built an AuditLogger class with log_event, query_events, and export_report
- Immutable audit logs use hash chaining to detect tampering
- Audit logging is essential for compliance (GDPR, HIPAA, SOX)

In the final notebook, you will generate compliance reports and model cards.